# Bölüm 8 · Tekrarlı Ölçümler ve Karma Desenler

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Kişi içi bağımlılığı analiz biriminden ayırır.
2. Zaman ve grup etkileşimini profil grafiğiyle yorumlar.
3. Tekrarlı ANOVA ile karma modelin koşullarını karşılaştırır.

**İleri rota:** Eksik izlem, kovaryans yapısı ve küçük örneklem çıkarımı; Bölüm 16 ile birlikte derinleştirilebilir.

**Ç1 · Çıkış sorusu:** 18 kişinin sekizer ölçümü 144 bağımsız kişi midir?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b08-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b08-tekrarli-olcumler'
modules=['uygulamalar/b08-tekrarli-olcumler/eksik-izlem']
expected_main=json.loads((root/"bolumler/b08/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 8 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b08/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 8 — görevler

Güncel kitap bölümünün etkinlik ve alıştırmalarıdır. Önce kendi çözümünüzü
üretin; ardından çözümleri kontrol edin. Kitap sayfa numaraları PDF sürümüne
göre değişebilir.

## D1–D3: Analizden Önce

1.  Neden 0. ve 1. günler ana analize alınmaz? Bu karar veriye bakmadan hangi belgeye dayanarak verilebilir?

2.  144 satırı bağımsız sayan tek yönlü ANOVA’nın hata serbestlik derecesi kaç olur? Tekrarlı ölçümler ANOVA’sında kaçtır? Fark neden önemlidir?

3.  Kontrol grubu olmadığında “yoksunluk tepki süresini uzatır” cümlesi hangi alternatif açıklamaları dışlayamaz?

Yanıtlar Ek F, Bölüm 8’dedir (s. kitaptaki ilgili sayfa).

<figure>
<embed src="assets_figures/tk-uyku-profilleri.pdf" />
<figcaption>Solda 18 kişinin temel günden (0) itibaren tepki süresi profilleri ve gün ortalamaları; sağda her kişi için ayrı doğrusal regresyonla bulunan eğimler ile karma etkiler modelinin kişiye özgü eğimleri (BLUP). Kesikli çizgi ortak (sabit) eğimdir. Kaynak: <code>lme4::sleepstudy</code>, 2.–9. günler; uygulama paketindeki <code>analiz.py</code>.</figcaption>
</figure>

Gün ortalamaları temel günde 265.4 ms’den 9. günde 350.9 ms’ye çıkar; standart sapmalar da 29.5 ms’den 67.0 ms’ye büyür. Şekil (kitaptaki ilgili başvuru) bu yayılmanın nedenini gösterir: kişilerin başlangıç düzeyleri farklıdır ve bazıları hızla kötüleşirken bazılarının profili neredeyse düzdür.

# Rehberli ve Bağımsız Etkinlikler

## G1–G4: Rehberli Çözüm

1.  Uyku verisinin kaynak tablosundaki SS ve sd değerlerinden MS ve F’yi yeniden hesaplayın. Toplam değişkenliğin yüzde kaçı kişi farklarıdır?

2.  $\hat\varepsilon_{GG}=.445$ ile düzeltilmiş sd’leri bulun. F değeri neden değişmez?

3.  $\eta^2_p$ ve $\eta^2_G$’yi hesaplayın. Neden bu kadar farklıdır?

4.  Doğrusal karşıtlık puanlarının ortalaması ve standart sapmasından $t$’yi hesaplayın. Bu test neden küresellik gerektirmez?

## B1–B3: Bağımsız Çözüm

1.  Orthodont kaynak tablosundan cinsiyet ve yaş için F’yi ve iki farklı hata terimini açıklayın.

2.  $\hat\varepsilon_{GG}=.867$ ile etkileşimin düzeltilmiş sd’lerini bulun. Özgün HF’nin neden 1’e kırpıldığını açıklayın.

3.  Genel etkileşim $p=.078$ iken büyüme hızı karşıtlığı $p=.019$ veriyor. İki sonucu nasıl birlikte raporlarsınız?

## H1: Hatalı Raporu Onarma

> “144 ölçümü tek yönlü ANOVA’ya koyduk. Mauchly anlamlı çıktığı için ANOVA yapılamaz. Kısmi eta-kare .47 olduğuna göre gün, bağımsız gruplar deneyindeki kadar büyük bir etki. Etkileşim anlamsız olduğundan kızlar ve erkekler aynı hızla büyüyor. En büyük farkı veren 2. ve 7. günleri seçip eşli t yaptık.”

Beş iddiayı düzeltin.

## P1: Tekrarlı Ölçüm Çalışması Planı

Bir müdahalenin dört haftalık izlem çalışması için tasarımı yazın: kontrol grubu, ölçüm zamanları, sıra etkileri, önceden belirlenmiş karşıtlıklar (örneğin grup $\times$ doğrusal zaman), küresellik kararı, eksik veri stratejisi (ANOVA mı karma model mi), etki büyüklüğü türü ve raporlanacak tablolar. Rubrik: tasarım 25, analiz planı 30, varsayımlar 20, raporlama 15, yeniden üretim 10 puan.

# Alıştırmalar

## Kolay Düzey

1.  Denek içi desenin bağımsız gruplar desenine göre temel üstünlüğü nedir?

2.  Küresellik varsayımı neyi ifade eder? İki ölçümlü desende neden sorun olmaz?

3.  Greenhouse–Geisser düzeltmesi F testinde neyi değiştirir?

4.  Uzun ve geniş veri biçimi arasındaki fark nedir?

5.  Karma (arası–içi) desende kaç ayrı hata terimi vardır?

## Orta Düzey

1.  Beş ölçümlü bir desende $\hat\varepsilon_{GG}=.62$, $F(4,76)=3.10$ bulunmuştur. Düzeltilmiş sd’leri yazın ve kararın nasıl değişebileceğini açıklayın.

2.  Bir araştırmacı $\eta^2_p=.40$ bulmuş ve bağımsız gruplar çalışmasındaki $\eta^2=.10$ ile karşılaştırıyor. Uyarınız nedir?

3.  Üç koşullu bir deneyde koşul sırası tüm katılımcılarda aynıdır. Hangi sorun doğar, nasıl önlenir?

4.  Dört ölçümlü bir çalışmada 40 katılımcının 9’unun bir ölçümü eksiktir. Tekrarlı ölçümler ANOVA’sı ile karma model arasında nasıl seçim yaparsınız?

5.  Karma desende etkileşim anlamlıdır. Ana etkileri nasıl yorumlarsınız?

## İleri Düzey

1.  Çok değişkenli yaklaşım ile GG düzeltmeli tek değişkenli test hangi koşullarda farklı sonuç verir? Hangisini ne zaman seçersiniz?

2.  Karma modelde rastgele eğim varyansı sıfıra yakın bulunursa ne sonuç çıkarırsınız?

3.  Genel etkileşim testi yerine önceden belirlenmiş tek sd’li karşıtlık kullanmanın gerekçesini ve riskini tartışın.

4.  Kişiler arası kovaryans matrisleri gruplar arasında belirgin biçimde farklıysa karma desen ANOVA’sı yerine ne yapılabilir?

5.  Güçlü bir tekrarlı ölçüm raporunda bulunması gereken en az beş unsuru yazın.


## Revizyon 11: eksik izlem ve aynı hedef

## T1–T3: Eksik İzlem ve Hedef Denetimi

1.  Özgün veride 144 tam ölçüm varken maskede 104 ölçüm/18 kişi, tam kişilerde 56 ölçüm/7 kişi kalmasını açıklayın. Üç satırda neden aynı doğrusal model kullanıldı? Tam-veri eğimi bilinen anakütle gerçeği midir?

2.  Maskenin MAR türü olduğu hangi tasarım bilgisinden gelir? Karma modelin yakınsaması, MAR veya doğru kovaryans kanıtı mıdır? 18 kişiyle normal Wald aralığı ve kişi eğim dağılımı aralığını ayırın.

3.  Dört yaş için Holm ve Bonferroni sonuçlarını hesaplayın. Bu kararlar genel etkileşim ve doğrusal eğim karşıtlığı hakkında neden aynı iddiayı üretmez? Karşılaştırma ailesinin ne zaman tanımlanması gerekir?


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b08/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Mevcut ölçümler veya yalnız tam kişiler yaklaşımını seçin; eğim, kişi sayısı ve kayıp varsayımını birlikte karşılaştırın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 8,
    "gorev": 'Mevcut ölçümler veya yalnız tam kişiler yaklaşımını seçin; eğim, kişi sayısı ve kayıp varsayımını birlikte karşılaştırın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":8,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b08-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b08-tekrarli-olcumler',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)